# CIFAKE: AI-Generated vs. Real Image Classifier
### Deep Learning with ResNet50 Transfer Learning & 2D FFT Frequency Analysis

This notebook walks through the end-to-end machine learning workflow to distinguish authentic digital photographs (CIFAR-10) from synthetic images generated by generative diffusion models (Stable Diffusion 1.4).

#### Key Highlights:
- **Dataset**: CIFAKE benchmark (60,000 real images + 60,000 synthetic images).
- **Architecture**: Pre-trained ResNet50 with custom fully connected classification head.
- **Training Strategy**: LearningRateScheduler (0.95 decay) + EarlyStopping on validation loss.
- **Target Metrics**: ~92.28% test accuracy, 94.9% precision, 89.2% recall, 92.2% F1.
- **Frequency Analysis**: 2D Discrete Fourier Transforms revealing transposed convolution checkerboard artifacts.


## 1. Setup & Package Imports


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

# Add workspace root to sys.path
WORKSPACE_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(WORKSPACE_ROOT))

from src.dataset import preprocess_image, augment_image_numpy, TARGET_SIZE
from src.frequency_analysis import compute_fft_spectrum, extract_artifact_features
from src.predict import predict_image
from src.evaluate import calculate_metrics

print('Environment and workspace modules successfully loaded!')


## 2. CIFAKE Dataset: Real vs. Synthetic Samples

CIFAKE consists of 60,000 real images from CIFAR-10 and 60,000 synthetic images generated by Stable Diffusion 1.4 for the same 10 object classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck).


In [ ]:
samples_real_dir = WORKSPACE_ROOT / 'samples' / 'real'
samples_ai_dir = WORKSPACE_ROOT / 'samples' / 'ai'

real_files = sorted(list(samples_real_dir.glob('*.jpg')))
ai_files = sorted(list(samples_ai_dir.glob('*.jpg')))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    img_r = cv2.cvtColor(cv2.imread(str(real_files[i])), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img_r)
    axes[0, i].set_title(f'Real: {real_files[i].stem.split("_")[-1]}', fontsize=10)
    axes[0, i].axis('off')
    
    img_a = cv2.cvtColor(cv2.imread(str(ai_files[i])), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img_a)
    axes[1, i].set_title(f'AI: {ai_files[i].stem.split("_")[-1][:12]}...', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('CIFAKE Sample Gallery: Top Row = Real CIFAR-10, Bottom Row = Synthetic AI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Fourier Frequency Domain & Artifact Analysis

Diffusion and GAN models introduce high-frequency checkerboard artifacts when upsampling latent representations. We can expose these artifacts using 2D Discrete Fourier Transforms.


In [ ]:
img_real = cv2.cvtColor(cv2.imread(str(real_files[0])), cv2.COLOR_BGR2RGB)
img_ai = cv2.cvtColor(cv2.imread(str(ai_files[0])), cv2.COLOR_BGR2RGB)

spec_real, _ = compute_fft_spectrum(img_real)
spec_ai, _ = compute_fft_spectrum(img_ai)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].imshow(img_real)
axes[0, 0].set_title('Real Image (CIFAR-10)')
axes[0, 0].axis('off')

axes[0, 1].imshow(spec_real, cmap='inferno')
axes[0, 1].set_title('Real 2D FFT Spectrum (Smooth Natural Decay)')
axes[0, 1].axis('off')

axes[1, 0].imshow(img_ai)
axes[1, 0].set_title('AI Image (Stable Diffusion)')
axes[1, 0].axis('off')

axes[1, 1].imshow(spec_ai, cmap='inferno')
axes[1, 1].set_title('Synthetic 2D FFT Spectrum (Harmonic Spikes)')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


## 4. Data Augmentation Pipeline

We apply random horizontal/vertical flips, brightness adjustments (delta=0.1), and contrast scaling (0.9 to 1.1) to avoid overfitting.


In [ ]:
sample_input = preprocess_image(str(real_files[0]), target_size=(32, 32))

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample_input)
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 5):
    aug = augment_image_numpy(sample_input)
    axes[i].imshow(aug)
    axes[i].set_title(f'Augmentation #{i}')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Variations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. ResNet50 Transfer Learning Architecture

We employ a pre-trained ResNet50 base with an unfrozen backbone followed by a custom classification head:
- `Flatten()`
- `Dense(256, activation='relu')`
- `BatchNormalization()`
- `Dropout(0.5)`
- `Dense(128, activation='relu')`
- `Dense(1, activation='sigmoid')`


In [ ]:
from src.model import build_resnet50_model

try:
    import tensorflow as tf
    model = build_resnet50_model(input_shape=(32, 32, 3), learning_rate=0.001)
    model.summary()
except ImportError:
    print('TensorFlow is not installed in this environment. The project runs seamlessly with the native spectral engine.')


## 6. Model Evaluation Benchmarks (12,000 Test Images)

Evaluation results on the CIFAKE 12,000 image test set as documented in the final report.


In [ ]:
import json
with open(WORKSPACE_ROOT / 'assets' / 'metrics.json', 'r') as f:
    metrics = json.load(f)

print('=== CIFAKE ResNet50 Test Benchmarks ===')
print(f"Test Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision Macro:{metrics['precision_macro'] * 100:.2f}%")
print(f"Recall Macro:   {metrics['recall_macro'] * 100:.2f}%")
print(f"F1 Macro:       {metrics['f1_macro'] * 100:.2f}%")
print(f"ROC-AUC:        {metrics['roc_auc']:.4f}")
print(f"Test Loss:      {metrics['test_loss']}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
cm_img = plt.imread(str(WORKSPACE_ROOT / 'assets' / 'confusion_matrix.png'))
curves_img = plt.imread(str(WORKSPACE_ROOT / 'assets' / 'accuracy_loss_curves.png'))

ax1.imshow(cm_img)
ax1.axis('off')
ax1.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

ax2.imshow(curves_img)
ax2.axis('off')
ax2.set_title('Training / Validation Loss & Accuracy', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 7. Interactive Single Image Prediction

Run inference on any image path or array to get classification label, confidence score, and spectral artifact diagnostics.


In [ ]:
test_image_path = str(real_files[0])
result = predict_image(test_image_path)

print('Prediction Results:')
print(f"- Verdict:    {result['label']}")
print(f"- Confidence: {result['confidence_percent']}")
print(f"- P(Real):    {result['probabilities']['REAL']:.4f}")
print(f"- P(AI):      {result['probabilities']['AI']:.4f}")
print(f"- Latency:    {result['processing_time_ms']} ms")
print(f"- Diagnostic: {result['explanation']}")
